# 4.12 · 随机森林回归 / Random Forest Regression

> **课程定位 / Where this fits**
> 第 12 课，**Part 4 · 监督学习：回归**。
> Lesson 12, **Part 4 · Supervised Regression**.
>
> 4.11 的单棵树**高方差**（数据一扰动就给出很不同的预测）。随机森林的思路极简：**训练很多去相关的树，取平均**，把方差压下去。回归版和分类版(5.7)原理相同，只是预测从"投票"变成"平均"。它强力、稳健、几乎零调参，是表格回归的**安全默认**。
> A single tree (4.11) is **high-variance**. Random Forest's idea is simple: **train many de-correlated trees and average them** to crush the variance. The regression version mirrors classification (5.7), except predictions average instead of vote. Strong, robust, near-zero-tuning — the **safe default** for tabular regression.
>
> 💼 **实战/面试视角**："bagging 怎么降方差 / OOB / RF vs GBDT" 是集成模型必考。
> 💼 **Practical/interview angle:** "how bagging reduces variance / OOB / RF vs GBDT" — must-know ensemble questions.

> 💡 **面试相关 / Interview-relevant**
> - "bagging 怎么降方差 / 为什么树要去相关"（出镜率 ★★★★★）
> - "随机森林会不会过拟合（树越多越好吗）"（★★★★★）
> - "OOB 误差是什么"（★★★★）
> - "RF vs GBDT 区别"（★★★★★，并行降方差 vs 串行降偏差）
> - "特征重要性的偏差"（★★★★）

---

## 学习目标 / Learning Objectives

1. 实测**bagging 的方差缩减**。
   Empirically measure bagging's variance reduction.
2. 理解"树越多越好、不会过拟合"（对比 boosting）。
   Understand "more trees is safe, no overfitting" (vs boosting).
3. 用 **OOB** 做免费泛化估计。
   Use OOB for a free generalization estimate.
4. 读特征重要性并知道其偏差。
   Read feature importances and their bias.
5. 对比单树 / RF / 预告 GBDT，并验证不需缩放。
   Compare tree / RF / (preview) GBDT, and verify no scaling needed.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [bagging 方差缩减（实测）⭐](#2)
3. [树越多越好，不过拟合 ⭐](#3)
4. [OOB 免费验证 ⭐](#4)
5. [特征重要性 ⭐](#5)
6. [对比 + 不需缩放 + 小结](#6)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

单棵树聪明但"情绪化"：它把训练集里的噪声也学了进去，换一批数据就给出很不同的预测（**高方差**）。随机森林的解法是"**群体智慧**"：训练**很多棵彼此略有差异的树**，让它们的预测**平均**——单棵树的随机抖动会互相抵消，集体预测稳得多。
A single tree is smart but "moody": it absorbs training noise and gives very different predictions on a different sample (**high variance**). Random Forest's fix is the **wisdom of crowds**: train **many slightly-different trees** and **average** their predictions — each tree's random wobble cancels out, and the collective is far steadier.

怎么让树"彼此不同"（去相关）？两处随机（同 5.7）：每棵树用一个 **bootstrap 样本**，每次分裂只看**随机的部分特征**。继续用 **Diamonds**。
How to make trees differ (de-correlate)? Two randomnesses (as in 5.7): each tree uses a **bootstrap sample**, and each split considers only a **random subset of features**. Continuing with **Diamonds**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(8000, random_state=0).reset_index(drop=True)
feat_names = ["carat","depth","table","x","y","z"]
X = df[feat_names].values; y = df["price"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
print(f"Diamonds: {X.shape}")


<a id="2"></a>
## 2. bagging 方差缩减（实测）⭐ / Variance Reduction, Measured

直接证明 bagging 在干什么：对**同一个测试点**，用 50 个不同的 bootstrap 样本各训一次单树、再各训一次森林，看预测的**标准差**。单树的预测忽上忽下（高方差），森林的预测则稳定得多——这就是"平均去相关预测"的方差缩减。
A direct proof of what bagging does: for **one test point**, fit a single tree and a forest on each of 50 different bootstrap samples, and look at the **standard deviation** of predictions. The single tree's predictions swing wildly (high variance); the forest's are far steadier — variance reduction by averaging de-correlated predictions.


In [ ]:
x_test_pt = X_te[:1]                                # 固定一个测试点
single_preds, forest_preds = [], []
for s in range(50):
    idx = rng.choice(len(X_tr), len(X_tr), replace=True)   # bootstrap 重采样
    single_preds.append(DecisionTreeRegressor(max_depth=10, random_state=s).fit(X_tr[idx], y_tr[idx]).predict(x_test_pt)[0])
    forest_preds.append(RandomForestRegressor(n_estimators=30, max_depth=10, random_state=s, n_jobs=-1).fit(X_tr[idx], y_tr[idx]).predict(x_test_pt)[0])

print("对同一测试点, 50 次重训的预测标准差:")
print(f"  单棵树   single tree: std = {np.std(single_preds):.1f}  (高方差, 预测很不稳)")
print(f"  随机森林 forest:      std = {np.std(forest_preds):.1f}  (方差骤降, 预测稳定)")
print(f"→ 森林预测稳定约 {np.std(single_preds)/np.std(forest_preds):.1f} 倍 — bagging 方差缩减的直接证据")


<a id="3"></a>
## 3. 树越多越好，不过拟合 ⭐ / More Trees Is Safe

随机森林一个让人安心的性质：**树越多，性能越好然后饱和，绝不会因树多而过拟合**。因为更多树只是让"平均"更稳，不会增加模型对训练数据的拟合能力。所以 `n_estimators` 越大越安全，唯一代价是更慢。**这和 boosting(4.13) 正好相反**——boosting 迭代过多会过拟合。
A reassuring property: **more trees → better, then plateau, never overfitting from tree count**. More trees only stabilize the average; they don't increase the model's capacity to fit training data. So larger `n_estimators` is always safe, the only cost being speed. **This is the opposite of boosting (4.13)** — too many boosting iterations overfit.


In [ ]:
n_trees = [1, 5, 10, 25, 50, 100, 200]
test_r2 = [RandomForestRegressor(n_estimators=nt, random_state=0, n_jobs=-1).fit(X_tr, y_tr).score(X_te, y_te) for nt in n_trees]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_trees, test_r2, "o-", lw=2)
ax.set_xlabel("树数 n_estimators"); ax.set_ylabel("test R²")
ax.set_title("树数 vs 性能: 上升后饱和, 不会因树多而过拟合")
plt.tight_layout(); plt.show()
for nt, r in zip(n_trees, test_r2): print(f"  n_estimators={nt:<4} test R²={r:.4f}")
print("树越多越好(单调)然后饱和 → RF 不会因树多过拟合(对比 boosting 4.13 会); 越大越安全, 只是变慢")


<a id="4"></a>
## 4. OOB 免费验证 ⭐ / Out-of-Bag Estimate

每棵树的 bootstrap 只用了约 63% 的样本，剩下约 37% 是这棵树"没见过"的（**袋外, out-of-bag**）。用每个样本"没见过它的那些树"来预测它，就得到一个**几乎免费的泛化估计**——不用单独划验证集。设 `oob_score=True` 即可。
Each tree's bootstrap uses ~63% of samples, leaving ~37% "out-of-bag" (unseen) for that tree. Predicting each sample with only the trees that never saw it gives a **nearly free generalization estimate** — no separate validation set needed. Just set `oob_score=True`.


In [ ]:
rf_oob = RandomForestRegressor(n_estimators=200, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
print(f"OOB R² (免费, 没用 test):  {rf_oob.oob_score_:.4f}")
print(f"真正的 test R²:            {rf_oob.score(X_te, y_te):.4f}")
cv = cross_val_score(RandomForestRegressor(100, random_state=0, n_jobs=-1), X_tr, y_tr, cv=5).mean()
print(f"5-fold CV R²:             {cv:.4f}")
print("OOB ≈ test ≈ CV → OOB 是可靠的泛化估计, 且零额外成本(训练时顺便算出)")
print("💡 数据珍贵时, OOB 让你不必牺牲数据做验证集")


<a id="5"></a>
## 5. 特征重要性 ⭐ / Feature Importance

随机森林的特征重要性 = 所有树上该特征带来的平均方差下降。这里 carat 和尺寸 (x,y,z) 主导钻石价格，符合常识。**但要注意它有偏**：基于不纯度/方差的重要性偏向**高基数/连续特征**。更可靠的是**置换重要性 (permutation importance，5.7)**——打乱一列看性能掉多少。
RF feature importance = average variance reduction a feature brings across all trees. Here carat and dimensions (x,y,z) dominate diamond price, as expected. **But beware its bias**: impurity/variance-based importance favors **high-cardinality/continuous features**. More reliable is **permutation importance (5.7)** — shuffle a column and see how much performance drops.


In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print("随机森林特征重要性:")
print(imp.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
imp.plot(kind="barh", ax=ax); ax.invert_yaxis()
ax.set_title("RF 特征重要性: carat + 尺寸(x,y,z) 主导钻石价格")
plt.tight_layout(); plt.show()
print("⚠ 默认 impurity-based 重要性偏向高基数/连续特征 → 更可靠用 permutation importance(5.7)")


<a id="6"></a>
## 6. 对比 + 不需缩放 + 小结 / Comparison, No Scaling & Summary

对比单树和森林：森林显著更好（方差缩减）。随机森林是表格回归的"安全默认"——强、稳、几乎零调参；要再榨最后几个点就上 GBDT/XGBoost(4.13)。森林继承了树的**不需缩放**。
Compare single tree vs forest: the forest is clearly better (variance reduction). RF is the "safe default" for tabular regression — strong, stable, near-zero-tuning; squeeze the last points with GBDT/XGBoost (4.13). Forests inherit trees' **scale-invariance**.


In [ ]:
from sklearn.preprocessing import StandardScaler
print(f"{'模型 model':<22} {'test R²':>9}")
for name, m in [("单棵树 tree(depth=10)", DecisionTreeRegressor(max_depth=10, random_state=0)),
                ("随机森林 RF(200)",       RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1))]:
    m.fit(X_tr, y_tr); print(f"{name:<22} {m.score(X_te, y_te):>9.4f}")
print("随机森林显著优于单树(方差缩减); GBDT(4.13) 通常还能再高一截")

# 验证 RF 也不需缩放(继承自树) / RF inherits scale-invariance
raw = cross_val_score(RandomForestRegressor(100, random_state=0, n_jobs=-1), X_tr, y_tr, cv=3).mean()
sc  = cross_val_score(RandomForestRegressor(100, random_state=0, n_jobs=-1), StandardScaler().fit_transform(X_tr), y_tr, cv=3).mean()
print(f"\n不缩放={raw:.4f}, 缩放={sc:.4f} → RF 同样不需缩放(继承自树)")


```
随机森林 = bagging 树 + 特征随机, 预测取平均(分类是投票 5.7); 降方差
方差缩减(实测): 同一点 50 次重训, 森林预测 std 远小于单树
树越多越好然后饱和, 不会过拟合(对比 boosting 会) → n_estimators 越大越安全(只变慢)
OOB: ~37% 袋外样本, 免费泛化估计, ≈test≈CV
特征重要性=方差下降(有偏, 偏连续特征) → 用 permutation importance(5.7) 更可靠
不需缩放(继承自树); 强、稳、零调参 = 表格回归安全默认; 榨极致用 GBDT(4.13)
```

### 💡 面试速查 / Interview cheat-sheet
1. **bagging 降方差**: 平均去相关的树(实测 std 大降)。
   Bagging reduces variance: averaging de-correlated trees (std drops measurably).
2. **树越多越好不过拟合**(对比 boosting 会过拟合)。
   More trees never overfit (opposite of boosting).
3. **OOB ≈ 免费 CV**(~37% 袋外)。
   OOB ≈ free CV (~37% out-of-bag).
4. **特征重要性有偏** → permutation importance 更可靠。
   Impurity importance is biased → permutation importance is more reliable.
5. **RF=并行降方差, GBDT=串行降偏差**; RF 不需缩放。
   RF = parallel variance reduction, GBDT = sequential bias reduction; RF needs no scaling.

### 下一节 / Next
**4.13 梯度提升回归(GBDT/XGBoost)**——从 bagging 切到 boosting: 一棵接一棵地拟合残差, 串行降偏差, 表格数据之王。
**4.13 Gradient Boosting (GBDT/XGBoost)** — from bagging to boosting: fit residuals sequentially to reduce bias; the king of tabular data.
